# Celebal Technologies Summer Internship 2026
## Gold Layer: Business-Ready Data & KPIs
### FMCG Data Consolidation & Analytics Platform

**Objective:** Create Star Schema (Fact + Dimension tables) and calculate business KPIs from Silver layer data.

**Star Schema:**
- Fact Table: fact_sales
- Dimensions: dim_customer, dim_product, dim_region, dim_time

**KPIs:**
- Total Sales by Region
- Top Products by Revenue
- Customer Lifetime Value
- Company Performance Comparison

In [0]:

# GOLD LAYER: Business Ready Data
# Medallion Architecture - Layer 3

from pyspark.sql.functions import (
    col, year, month, quarter, dayofweek,
    sum as spark_sum, avg, count,
    round as spark_round, expr
)

print("=" * 60)
print("GOLD LAYER: Star Schema Creation Started")
print("=" * 60)

# Create Gold database
spark.sql("CREATE DATABASE IF NOT EXISTS gold_fmcg")

# Load Silver data
df_silver = spark.read.table("silver_fmcg.unified_sales")
print(f"Silver data loaded: {df_silver.count()} rows")

GOLD LAYER: Star Schema Creation Started
Silver data loaded: 10001 rows


In [0]:

# Dimension Tables


# DIM_CUSTOMER
spark.sql("DROP TABLE IF EXISTS gold_fmcg.dim_customer")
dim_customer = df_silver \
    .select("customer_id", "customer_name", "Segment", "Country") \
    .dropDuplicates(["customer_id"]) \
    .withColumnRenamed("Segment", "segment") \
    .withColumnRenamed("Country", "country")
dim_customer.write.format("delta").mode("overwrite").saveAsTable("gold_fmcg.dim_customer")

# DIM_PRODUCT
spark.sql("DROP TABLE IF EXISTS gold_fmcg.dim_product")
dim_product = df_silver \
    .select("product_name", "Category", "sub_category") \
    .dropDuplicates(["product_name"]) \
    .withColumnRenamed("Category", "category")
dim_product.write.format("delta").mode("overwrite").saveAsTable("gold_fmcg.dim_product")

# DIM_REGION
spark.sql("DROP TABLE IF EXISTS gold_fmcg.dim_region")
dim_region = df_silver \
    .select("Region", "city", "Country") \
    .dropDuplicates(["Region", "city"]) \
    .withColumnRenamed("Region", "region") \
    .withColumnRenamed("Country", "country")
dim_region.write.format("delta").mode("overwrite").saveAsTable("gold_fmcg.dim_region")

# DIM_TIME
spark.sql("DROP TABLE IF EXISTS gold_fmcg.dim_time")
dim_time = df_silver \
    .select("order_date") \
    .dropDuplicates(["order_date"]) \
    .withColumn("year", year(col("order_date"))) \
    .withColumn("month", month(col("order_date"))) \
    .withColumn("quarter", quarter(col("order_date"))) \
    .withColumn("day_of_week", dayofweek(col("order_date")))
dim_time.write.format("delta").mode("overwrite").saveAsTable("gold_fmcg.dim_time")

print("=== Dimension Tables Created ===")
print(f"dim_customer: {dim_customer.count()} rows")
print(f"dim_product:  {dim_product.count()} rows")
print(f"dim_region:   {dim_region.count()} rows")
print(f"dim_time:     {dim_time.count()} rows")

dim_customer.show(5)
dim_product.show(5)

=== Dimension Tables Created ===
dim_customer: 804 rows
dim_product:  1862 rows
dim_region:   593 rows
dim_time:     1252 rows
+-----------+-------------------+-----------+-------------+
|customer_id|      customer_name|    segment|      country|
+-----------+-------------------+-----------+-------------+
|   KL-16555|      Kelly Lampkin|  CORPORATE|United States|
|   JE-15475|     Jeremy Ellison|   CONSUMER|United States|
|   MS-17365|Maribeth Schnelling|   CONSUMER|United States|
|   ML-17755|         Max Ludwig|HOME OFFICE|United States|
|   JP-16135|     Julie Prescott|HOME OFFICE|United States|
+-----------+-------------------+-----------+-------------+
only showing top 5 rows
+--------------------+---------------+------------+
|        product_name|       category|sub_category|
+--------------------+---------------+------------+
|OIC Colored Binde...|Office Supplies|   Fasteners|
|Avery Durable Pol...|Office Supplies|     Binders|
|Belkin Grip Candy...|     Technology|      Phone

In [0]:

# Fact Table: FACT_SALES

spark.sql("DROP TABLE IF EXISTS gold_fmcg.fact_sales")

fact_sales = df_silver \
    .select(
        "order_id", "order_date", "customer_id",
        "product_name", "Region", "city",
        "Sales", "Quantity", "Discount", "Profit",
        "source_company"
    ) \
    .withColumnRenamed("Region", "region") \
    .withColumnRenamed("Sales", "sales_amount") \
    .withColumnRenamed("Quantity", "quantity") \
    .withColumnRenamed("Discount", "discount") \
    .withColumnRenamed("Profit", "profit") \
    .withColumn("profit_margin",
        spark_round(expr("try_divide(profit, sales_amount)") * 100, 2))

fact_sales.write.format("delta").mode("overwrite").saveAsTable("gold_fmcg.fact_sales")
print(f" fact_sales: {fact_sales.count()} rows")
fact_sales.show(5)

 fact_sales: 10001 rows
+--------------+----------+-----------+--------------------+-------+------------+------------+--------+--------+-------+--------------+-------------+
|      order_id|order_date|customer_id|        product_name| region|        city|sales_amount|quantity|discount| profit|source_company|profit_margin|
+--------------+----------+-----------+--------------------+-------+------------+------------+--------+--------+-------+--------------+-------------+
|CA-2014-115812|2014-06-09|   BH-11710|Mitel 5320 IP Pho...|   WEST| Los Angeles|     907.152|     6.0|     0.2|90.7152|     Company_A|         10.0|
|US-2017-156909|2017-07-16|   SF-20065|Global Deluxe Sta...|   EAST|Philadelphia|      71.372|     2.0|     0.3|-1.0196|     Company_A|        -1.43|
|CA-2015-115742|2015-04-18|   DP-13000|C-Line Peel & Sti...|CENTRAL|  New Albany|       38.22|     6.0|     0.0|17.9634|     Company_A|         47.0|
|CA-2016-111682|2016-06-17|   TB-21055|Novimex Turbo Tas...|   EAST|        

In [0]:
# KPI 1: Total Sales by Region
print("=== KPI 1: Total Sales by Region ===")
kpi_region = fact_sales \
    .groupBy("region") \
    .agg(
        spark_round(spark_sum("sales_amount"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        count("order_id").alias("total_orders"),
        spark_round(avg("profit_margin"), 2).alias("avg_profit_margin")
    ).orderBy("total_sales", ascending=False)

spark.sql("DROP TABLE IF EXISTS gold_fmcg.kpi_sales_by_region")
kpi_region.write.format("delta").mode("overwrite").saveAsTable("gold_fmcg.kpi_sales_by_region")
kpi_region.show()

# KPI 2: Top Products
print("=== KPI 2: Top 10 Products by Sales ===")
kpi_products = fact_sales \
    .groupBy("product_name") \
    .agg(
        spark_round(spark_sum("sales_amount"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        count("order_id").alias("total_orders")
    ).orderBy("total_sales", ascending=False)

spark.sql("DROP TABLE IF EXISTS gold_fmcg.kpi_product_performance")
kpi_products.write.format("delta").mode("overwrite").saveAsTable("gold_fmcg.kpi_product_performance")
kpi_products.show(10)

# KPI 3: Customer Lifetime Value
print("=== KPI 3: Top 10 Customers ===")
kpi_clv = fact_sales \
    .groupBy("customer_id") \
    .agg(
        spark_round(spark_sum("sales_amount"), 2).alias("lifetime_value"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        count("order_id").alias("total_orders")
    ).orderBy("lifetime_value", ascending=False)

spark.sql("DROP TABLE IF EXISTS gold_fmcg.kpi_customer_lifetime_value")
kpi_clv.write.format("delta").mode("overwrite").saveAsTable("gold_fmcg.kpi_customer_lifetime_value")
kpi_clv.show(10)

# KPI 4: Company Comparison
print("=== KPI 4: Sales by Source Company ===")
kpi_company = fact_sales \
    .groupBy("source_company") \
    .agg(
        spark_round(spark_sum("sales_amount"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        count("order_id").alias("total_orders")
    ).orderBy("total_sales", ascending=False)

spark.sql("DROP TABLE IF EXISTS gold_fmcg.kpi_company_comparison")
kpi_company.write.format("delta").mode("overwrite").saveAsTable("gold_fmcg.kpi_company_comparison")
kpi_company.show()

=== KPI 1: Total Sales by Region ===
+-----------+-----------+------------+------------+-----------------+
|     region|total_sales|total_profit|total_orders|avg_profit_margin|
+-----------+-----------+------------+------------+-----------------+
|       WEST|  713369.01|   107289.63|        3202|            21.67|
|       EAST|  671319.18|    91434.16|        2845|            16.84|
|    CENTRAL|  497800.87|     40150.5|        2323|           -10.01|
|      SOUTH|  388269.51|    46450.11|        1616|            16.48|
|NORTH INDIA|    13000.0|      1905.0|           5|             16.0|
|SOUTH INDIA|    12300.0|      2272.5|           4|            16.88|
| WEST INDIA|     9600.0|      1015.0|           5|             11.0|
| EAST INDIA|     4500.0|       675.0|           1|             15.0|
+-----------+-----------+------------+------------+-----------------+

=== KPI 2: Top 10 Products by Sales ===
+--------------------+-----------+------------+------------+
|        product_name